# We will work on the first few problems and then you work in small groups, write the queries
# Selected groups will present their solution and have a open discussion in class, e.g., different ways to write the queries

# Setup Spark environment to connect to the cluster

In [ ]:
# Documentation: https://spark.apache.org/docs

from pyspark import SparkContext, SparkConf
cf = SparkConf()
cf.set("spark.submit.deployMode","client")
# get the sparkcontext to communicate with the cluster
sc = SparkContext.getOrCreate(cf)

In [ ]:
# The entry point into all functionality in Spark is the SparkSession class.
#To create a basic SparkSession, just use SparkSession.builder:

from pyspark.sql import SparkSession
spark = SparkSession \
    .builder \
    .appName("Python Spark SQL basic example") \
    .config("spark.some.config.option", "some-value") \
    .getOrCreate()

## Read dataset

In [ ]:
path="/shared/CS-GY-6513/Wildlife"
ads_f = path + "/data_labeled_feb_7_2024.parquet"  #larger file
df_ads = spark.read.parquet(ads_f);

In [ ]:
# check the number of partitions
print("Parquet (larger file):", df_ads.rdd.getNumPartitions())

In [ ]:
# check the partition size
print(spark.conf.get("spark.sql.files.maxPartitionBytes"))   # 134217728 (128MB), by default


In [ ]:
# check the file size
import subprocess
print(subprocess.run(["hdfs", "dfs", "-du", "-h", ads_f], capture_output=True, text=True).stdout)

In [ ]:
# force the creation of 4 partitions
df_ads.repartition(4).rdd.getNumPartitions()

In [ ]:
# Note: repartition() is expensive; the more partitions, the higher the scheduling overheads, creates the
# many small files problem

## Caching

In [ ]:
import time

# time a count() cold start
start = time.time()
df_ads.count()
print(f"Count takes - cold start: {time.time() - start:.2f}s")

In [ ]:
# time a count() with no caching
start = time.time()
df_ads.count()
print(f"Count takes - warm start, no caching: {time.time() - start:.2f}s")

In [ ]:
# cache it -- this doesn't compute anything yet (still lazy
# df_ads will be cached the next time an action is applied to it
df_ads.cache()
# the FIRST action after cache() is what actually materializes it
start = time.time()
df_ads.count()
print(f"First action after cache() (materializes it): {time.time() - start:.2f}s")


In [ ]:

# reading from the cache
start = time.time()
df_ads.count()
print(f"Second action after cache() (materializes it): {time.time() - start:.2f}s")


In [ ]:
#  drop the cached data from memory/disk and revert to needing a fresh scan on the next action.
df_ads.unpersist(blocking=True)  # ensure cache is flushed!


# clear the whole cache
# spark.catalog.clearCache()


In [ ]:
# (useDisk, useMemory, useOffHeap, deserialized, replication)
# useDisk = True — Spark is allowed to spill this data to disk if it doesn't fit in memory
# useMemory = True — Spark will try to keep it in memory
# useOffHeap = False — not using off-heap storage (a separate memory region outside the JVM heap, used for StorageLevel.OFF_HEAP)
# deserialized = True — stored as live Java/Python objects, not a serialized byte format (faster to read, but takes more memory than serialized storage)
# replication = 1 — one copy of each partition, no redundancy across nodes
df_ads.storageLevel


In [ ]:
df_ads.is_cached


In [ ]:
# Note: the very first execution of anything in Spark is confounded by JVM/codegen/OS-cache warm-up,
# regardless of whether Spark-level caching is involved.

## Explore the data


In [ ]:
# check the schema of the data
df_ads.printSchema()

### Querying the data using DataFrames


In [ ]:
# check the data contents
df_ads.take(5)

In [ ]:
# df_ads is a dataframe and it can be manipulated using dataframe operations
# very similar to Pandas
df_ads.select("title").show()

In [ ]:
# what is the SQL query equivalent to the command below?
df_ads.select(df_ads['title'], df_ads['price']).show()


In [ ]:
df_ads.select("title","price").filter(df_ads['price'] > 1000).show()


## Using SQL

In [ ]:
# If you are like me, I very much prefer to write SQL queries
# To do so, register the DataFrame as a SQL temporary view
df_ads.createOrReplaceTempView("ads")

# Note: you can translate SQL to Pandas:
# https://sql2pandas.pythonanywhere.com/


In [ ]:
spark.sql("SELECT title,price,final_label FROM ads").show()


<b> Be careful with parsing! </b> Spaces, commas, and quotes in values, as well as values that are too long or take multiple lines can cause problems.
Sometimes csv files that load correctly in Pandas do not load correctly on Spark.


### Querying the data using SQL
![image.png](attachment:image.png)

In [ ]:
spark.sql("DESCRIBE ads").show()


In [ ]:
# Here's a very simple query
spark.sql("SELECT title from ads").show()


In [ ]:
spark.sql("SELECT id, animal_name, animal_original from ads").show()


In [ ]:
# List sellers and product label if product is animal related
spark.sql("SELECT seller,final_label FROM ads where final_label=1").show()


In [ ]:
# for each seller, list how many animal related products they sell
spark.sql("SELECT seller,count(*) FROM ads WHERE final_label=1 GROUP BY seller").show()


In [ ]:
# look at the physical plan -- note the shuffle/exchange step
df_ads.unpersist(blocking=True)  # ensure cache is flushed!

spark.sql("SELECT seller, count(*) FROM ads WHERE final_label=1 GROUP BY seller").explain()


In [ ]:
result = spark.sql("SELECT seller, count(*) FROM ads WHERE final_label=1 GROUP BY seller")
result.collect()
result.explain()
print(result.rdd.getNumPartitions())

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", 8)


In [ ]:
spark.sql("SELECT seller, count(*) FROM ads WHERE final_label=1 GROUP BY seller").explain()

In [ ]:
result = spark.sql("SELECT seller, count(*) FROM ads WHERE final_label=1 GROUP BY seller")
result.collect()
result.explain()
print(result.rdd.getNumPartitions())

### Note that the result of a SparkSQL query is a dataframe


In [ ]:
# for each seller, list how many animal related products they sell -- in descending order
# sparksql does not support TOP(K) queries but it does support LIMIT
top_sellers = spark.sql("SELECT seller,count(*) AS num_ads FROM ads WHERE final_label=1 GROUP BY seller ORDER BY num_ads DESC LIMIT 5")
top_sellers.show()



In [ ]:
# Note: dirty and incomplete data

In [ ]:
# Exercise: What animals does the top seller sell?
# list all distinct animals mentioned both in animal_name and animal_original


In [ ]:
# Exercise: List the seller and title for ads that mention wolf
# do not truncate the results



In [ ]:
# Exercise: for each seller, list how many non-animal related products they sell



In [ ]:
# Exercise: for each seller, list how many non-animal related products they sell
# To show all sellers, not just top 20, tell SQLSpark the number of rows
# you want displayed


In [ ]:
#Exercise" List sellers that sell both animal-related and non-animal products


In [ ]:
# Exercise
#List sellers that sell both animal-related and non-animal products, including the number of animal-related and non-animal ads


In [ ]:
# Exercise: For each animal, list the number of ads in which they appear


In [ ]:
# Exercise
# How many sellers sell ostrich-related products?



In [ ]:
# List the the title, seller, and price of all products related to Ostrich


In [ ]:
# What is the average price of Ostrich-related products?


In [ ]:
# For each seller that sells ostrich-related products,
# What is the average price of  products?


### Saving results as files on HDFS

In [ ]:
ostrich_sellers.select("*").write.save("ostrich-sellers.csv", format="csv")


# after exiting pyspark, to move the written file from HDFS to your home directory, you type
# hfs -getmerge sailorscsv.csv sailorscsv.csv
